# Unit 12 exercises: sequence-aware retrieval (what you read *last* matters)

Run the recsys data generators before you begin. Every path the course has built so far treats a reader's history as a **bag** of books: Unit 4's item-item CF sums the similarity to *every* book a reader has liked, in no particular order. This unit asks whether the **order** matters -- does the book a reader finished **last** say more about the next one than a book from months ago? You will drill the three ideas of the lesson: **recency** (query with the last *k* distinct books, `LastKItemItemPath`), **item transitions** (a first-order "what do people read *right after* X" table, `TransitionRetrievalPath`) checked by an **order-shuffled control**, and **causal self-attention** (the tiny SASRec-style `SasrecRetrievalPath`), read honestly.

**These exercises run on the session log (`sessions.csv.gz`), not the main interaction log.** It is a separate synthetic reader population (1,500 readers, 1,419 of them scored on `val`) with real order effects -- series next-volumes, author streaks, genre moods. Every number here is "on the session log" and is **not comparable** to the numbers of Units 2-11. Compare paths only *within* this log.

Everything is graded on the **validation** split (`val`); each reader's **seen** set is their train-positive history, and only train positives ever enter a learned statistic or a stored history. The sealed `test` holdout stays sealed -- no cell here may touch it. The series table (`series.csv.gz`) is a **diagnostic lens only**, never a model input. Each answer is left for you to complete; no solutions live here.

A speed note. The numpy paths (`LastKItemItemPath`, `TransitionRetrievalPath`) fit in about a second, and one scoreboard pass takes a few seconds. The SASRec model takes about **45 s** to train, so Exercise 5 fits it **exactly once**, in its own cell, and later exercises reuse that fitted path. Do not fit a second SASRec anywhere in this notebook.

In [ ]:
import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LastKItemItemPath,
    PopularityRetrievalPath,
    SasrecRetrievalPath,
    TransitionRetrievalPath,
    first_unseen_val,
    load_series,
    load_train_sequences,
    reader_hits,
    run_blended_scoreboard,
)
from bookrec.protocol import BaseRetrievalPath, Candidate

K = 10                           # every scoreboard reports at the top-10 cutoff
SHUFFLE_SEED = 20260930 + 1000   # the order-shuffled control's seed (the lesson's)

# --- the SESSION log (Unit 12 only): rows carry a timestamp, so each history has an order ---
data_dir = bookrec.generated_dir()
catalog_ids = sorted(bookrec.load_catalog(data_dir / "catalog.csv.gz"))   # 2000 books
col_of = {item_id: col for col, item_id in enumerate(catalog_ids)}      # book -> matrix column
sessions_path = data_dir / "sessions.csv.gz"
rows = pd.read_csv(sessions_path).to_dict("records")   # reader_id, item_id, timestamp, split, label, ...

# Each reader's ORDERED train-positive history (oldest first; val/test never enter it).
sequences = load_train_sequences(sessions_path)
# Each scored reader's time-ordered FIRST val positive that is not a re-read (the next-item target).
next_item = first_unseen_val(sessions_path)


def board(path):
    # hit@10 + catalog coverage on `val` via the single-path blended scoreboard (reproduces hit@k).
    result = run_blended_scoreboard({path.name: path}, sessions_path, catalog_ids=catalog_ids, k=K, pool=K)
    return result.hit_rate_at_k, result.catalog_coverage


# The two baselines every exercise compares against (both bag models, from Units 2 and 4).
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
item_item_hit, item_item_cov = board(item_item)       # ~0.480 / ~0.336 on the session log
popularity_hit, popularity_cov = board(popularity)    # ~0.113 / ~0.008

## Exercise 1 - The last-k dial: from "remember everything" to "only the last book"

`LastKItemItemPath(k_last)` keeps Unit 4's item-item cosine exactly, but queries it with only the reader's last `k_last` **distinct** train books (a re-read counts once, at its most recent position). `k_last=None` queries with *every* distinct book -- the bag -- so Unit 4's path is one end of the same dial.

Fit the path at `k_last` = 1, 3, 5 and `None` (all), and board each on `val` (hit@10 and coverage). Confirm that `None` reproduces item-item's hit@10 exactly (~0.480), that recency lifts it (last-3 ~0.551), that the sweep peaks at 5 (~0.572), and that `k_last=1` is *too* short (it falls well below item-item). In two or three sentences, explain the trade the dial makes: what does a short window gain, and what does a window of one throw away?

In [ ]:
k_values = [1, 3, 5, None]
last_k = None     # TODO: {k: LastKItemItemPath(k).fit(rows, catalog=catalog_ids) for k in k_values}
sweep = None      # TODO: {k: board(last_k[k]) for k in k_values}   -- {k: (hit@10, coverage)}

none_is_item_item = None   # TODO: bool - abs(sweep[None][0] - item_item_hit) < 1e-12   (the bag end of the dial)
recency_lifts = None       # TODO: bool - sweep[3][0] > item_item_hit                  (~0.551 vs ~0.480)
best_k = None              # TODO: the k in k_values with the highest hit@10           (expect 5)
one_is_too_short = None    # TODO: bool - sweep[1][0] < item_item_hit                  (~0.36: one book is too little)

dial_explanation = "TODO"  # 2-3 sentences: a short window queries with the books that predict the
                           # NEXT read (recent series volumes, the current author/genre mood) and drops
                           # stale tastes; a window of one throws away the rest of the recent context,
                           # so the sweet spot is a handful of recent distinct books

## Exercise 2 - Count item transitions by hand, then reveal `TransitionRetrievalPath`

A first-order **transition** (Markov) model asks: across all readers, how often was book $b$ read *right after* book $a$? Count it as a matrix $T$ with $T[a, b]$ = the number of consecutive train pairs $(a, b)$, summed over every reader's ordered history in `sequences`. A reader is then scored by summing the $T$ rows of their last 5 books: a candidate scores high when it often *follows* what they just read.

Build $T$ by hand as a `(2000, 2000)` float array (rows: *from*, columns: *to*, indexed by `col_of`), walking each history's consecutive pairs with `zip(history[:-1], history[1:])`. Confirm it holds exactly one count per consecutive pair. Reveal `TransitionRetrievalPath(5)`, fit it on `rows`, and confirm its `transitions` matrix equals yours exactly. Then pick the first reader (smallest id) with at least 6 train books, compute their by-hand score vector from the $T$ rows of their last 5 books, and confirm every candidate the path retrieves for them (`retrieve(reader, {"seen": ...}, K)`) carries exactly your by-hand score. Finally, board the path (~0.516). In two sentences, say why only **consecutive train** pairs may be counted -- what goes wrong if a reader's last train book -> first val book pair enters $T$?

In [ ]:
T = None                   # TODO: np.zeros((len(catalog_ids), len(catalog_ids)))
# TODO: for each history in sequences.values(), for each consecutive pair (a, b) in
#       zip(history[:-1], history[1:]): T[col_of[a], col_of[b]] += 1.0
n_consecutive_pairs = None # TODO: sum(len(history) - 1 for history in sequences.values())
one_count_per_pair = None  # TODO: bool - int(T.sum()) == n_consecutive_pairs

transition = None          # TODO: TransitionRetrievalPath(5).fit(rows, catalog=catalog_ids)
matches_path = None        # TODO: bool - np.array_equal(T, transition.transitions)

reader = None              # TODO: the smallest reader id in sequences with len(sequences[reader]) >= 6
history = None             # TODO: sequences[reader]   -- oldest first
hand_scores = None         # TODO: T[[col_of[item] for item in history[-5:]]].sum(axis=0)   -- one score per book
path_top = None            # TODO: transition.retrieve(reader, {"seen": set(history)}, K)
scores_match = None        # TODO: bool - every c in path_top has abs(hand_scores[col_of[c.item_id]] - c.score) < 1e-9

transition_hit, transition_cov = None, None   # TODO: board(transition)   -- ~0.516

leakage_explanation = "TODO"  # 2 sentences: T may count only consecutive TRAIN positives; the last-train
                              # -> first-val step is exactly what we are trying to predict, so counting it
                              # (or any val/test read) leaks the answer into the model

## Exercise 3 - The order-shuffled control: is it the order, or just the set of books?

A transition score of ~0.516 *looks* good, but a raw number proves nothing about order: maybe the model only learned which books go together, whatever their order. The test is a **control**. `TransitionRetrievalPath(5, shuffle_seed=SHUFFLE_SEED)` randomly permutes every reader's history -- the same books, a random order -- and uses that shuffled history **both** to count $T$ **and** to pick the "last 5" query books. If order carries the lift, the shuffled copy should collapse.

Fit the shuffled control and board it. Confirm the ordered path beats it by at least 1.3x and by at least +0.10 (expect ~0.516 vs ~0.333). Check that the control really is a permutation: read both paths' stored `histories` **once each** (the property returns a full copy), then confirm every reader keeps exactly the same books (`sorted(...)` equal) while at least one reader's order changed. Then **predict before you run it**: will shuffling hurt the *recency* path (`LastKItemItemPath(3, shuffle_seed=SHUFFLE_SEED)`) as well, or does it only need the set of books? Write your prediction first, then measure it. In three or four sentences, explain why the control must shuffle both the learning and the query histories, and what your last-3 result says about where recency gets its information.

In [ ]:
transition_shuffled = None   # TODO: TransitionRetrievalPath(5, shuffle_seed=SHUFFLE_SEED).fit(rows, catalog=catalog_ids)
shuffled_hit, shuffled_cov = None, None   # TODO: board(transition_shuffled)   -- ~0.333

order_ratio = None           # TODO: transition_hit / shuffled_hit                  -- ~1.55
order_evidence = None        # TODO: bool - order_ratio >= 1.3 and transition_hit >= shuffled_hit + 0.10

ordered_histories = None     # TODO: transition.histories            (read ONCE)
shuffled_histories = None    # TODO: transition_shuffled.histories   (read ONCE)
same_books = None            # TODO: bool - every reader r: sorted(shuffled_histories[r]) == sorted(ordered_histories[r])
order_changed = None         # TODO: bool - some reader r: shuffled_histories[r] != ordered_histories[r]

last3_prediction = "TODO"    # write this BEFORE running the next two lines: will shuffling hurt last-3 CF?
last3_shuffled = None        # TODO: LastKItemItemPath(3, shuffle_seed=SHUFFLE_SEED).fit(rows, catalog=catalog_ids)
last3_shuffled_hit = None    # TODO: board(last3_shuffled)[0]   -- compare with sweep[3][0] from Exercise 1

control_explanation = "TODO"  # 3-4 sentences: the control keeps the item SET and destroys only the ORDER,
                              # so the drop is the evidence order matters; it must shuffle BOTH learning and
                              # query (a half-shuffle is a weaker, misleading control); last-3 also collapses
                              # because "which books are last" is itself order information

## Exercise 4 - Next-item hit vs any-item hit

The scoreboard's hit@10 counts a hit when **any** of a reader's held-out `val` books lands in their top 10. A sequence model makes a sharper claim: it predicts the **next** book. `next_item` (setup cell) holds, for each scored reader, their time-ordered **first** `val` book that is not a re-read. **Next-item hit@10** is the fraction of those readers whose next book is in their top 10 (with their train history as the `seen` set, ranked with `bookrec.rank(..., n=K, exclude=seen)`).

Write `next_item_hit(path)`, confirm it covers all 1,419 readers, and measure it for item-item, last-5 (`last_k[5]` from Exercise 1) and transition (Exercise 2). Confirm every path's next-item hit is **below** its any-item hit (expect item-item ~0.245, last-5 ~0.388, transition ~0.298). Then compare the order paths' gain over item-item on the two metrics as **ratios**. In two or three sentences, explain why next-item hit is the stricter metric, and why the order-aware paths gain *relatively* more on it.

In [ ]:
def next_item_hit(path):
    # Fraction of readers in `next_item` whose next book is in the path's top K (train history = seen).
    return None   # TODO: for each (reader, target) in next_item.items(): seen = set(sequences.get(reader, ()));
                  #       top = bookrec.rank(path.retrieve(reader, {"seen": seen}, K), n=K, exclude=seen);
                  #       record 1.0 if target is among the top item ids else 0.0; return float(np.mean(...))


n_next_readers = None            # TODO: len(next_item)   -- 1419
next_item_item = None            # TODO: next_item_hit(item_item)    -- ~0.245
next_last5 = None                # TODO: next_item_hit(last_k[5])    -- ~0.388
next_transition = None           # TODO: next_item_hit(transition)   -- ~0.298

next_below_any = None            # TODO: bool - next_item_item < item_item_hit and next_last5 < sweep[5][0]
                                 #       and next_transition < transition_hit
any_ratio_transition = None      # TODO: transition_hit / item_item_hit      -- any-item gain over the bag
next_ratio_transition = None     # TODO: next_transition / next_item_item    -- next-item gain over the bag
order_gains_more_on_next = None  # TODO: bool - next_ratio_transition > any_ratio_transition

next_item_explanation = "TODO"   # 2-3 sentences: any-item hit forgives the model for finding ANY held-out
                                 # book; next-item asks for exactly the one read next, so it is stricter;
                                 # order paths aim at "what comes next", so their edge over the bag grows

## Exercise 5 - One SASRec fit, read honestly against item-item CF

`SasrecRetrievalPath()` is the lesson's tiny causal self-attention model: 1 block, 1 head, dim 32, the last <= 50 books, the unit's fast 30-epoch recipe, seeded and single-threaded so two fits give the same top-10 lists. Train it **once** on `rows` (about 45 s; the fit cell does nothing else) and board it.

Then compare it with item-item CF **reader by reader**. `reader_hits(path, sessions_path, catalog_ids=catalog_ids, k=K)` returns each scored reader's hit@10 (0.0 or 1.0) on exactly the scoreboard's cohort, so its mean is the path's hit@10. Take the per-reader differences SASRec minus item-item over the same readers; their mean is the gap, and their **paired standard error** is `np.std(diffs, ddof=1) / np.sqrt(len(diffs))`. Expect SASRec at about 0.423, **trailing** item-item by about 0.06 (-0.057, SE ~0.012), and the order paths by more.

You will **not** fit a shuffled SASRec here (it would cost another 45 s). The unit's milestone does that in CI: ordered **0.423** vs order-shuffled **0.383**, a gap of **+0.040** at **3.8 paired standard errors**. Read those cited numbers in your answer. In three or four sentences: does SASRec learn order (use the cited control), does it win on this data (use your own gap), and why can a simple counting model beat a small attention model here?

In [ ]:
# The ONE SASRec fit in this notebook (~45 s). Do not refit it below -- reuse `sasrec`.
sasrec = None   # TODO: SasrecRetrievalPath().fit(rows, catalog=catalog_ids)

In [ ]:
sasrec_hit, sasrec_cov = None, None   # TODO: board(sasrec)   -- ~0.423

sasrec_by_reader = None      # TODO: reader_hits(sasrec, sessions_path, catalog_ids=catalog_ids, k=K)
item_item_by_reader = None   # TODO: reader_hits(item_item, sessions_path, catalog_ids=catalog_ids, k=K)
same_cohort = None           # TODO: bool - sasrec_by_reader.keys() == item_item_by_reader.keys()
diffs = None                 # TODO: np.array([sasrec_by_reader[r] - item_item_by_reader[r] for r in sorted(item_item_by_reader)])
gap_to_item_item = None      # TODO: float(diffs.mean())                               -- ~ -0.057
paired_se = None             # TODO: float(np.std(diffs, ddof=1) / np.sqrt(len(diffs)))  -- ~0.012
trails_item_item = None      # TODO: bool - gap_to_item_item < 0
trails_order_paths = None    # TODO: bool - sasrec_hit < transition_hit and sasrec_hit < sweep[3][0]

# The ordered-vs-shuffled SASRec control is MEASURED by this unit's milestone (CI), not refit here.
CITED_SASREC_ORDERED, CITED_SASREC_SHUFFLED = 0.423, 0.383   # milestone: +0.040, 3.8 paired SE

sasrec_reading = "TODO"  # 3-4 sentences: the cited control (0.423 vs 0.383, +0.040 at 3.8 SE) shows SASRec
                         # DOES learn order; your own gap shows it TRAILS item-item (~ -0.06, several SE) and
                         # the simple order paths by more -- it neither wins nor ties at this recipe; the order
                         # effects here are short-range and item-to-item, which a counting model captures
                         # directly, while a tiny attention model cannot use its extra flexibility

## Exercise 6 - Coverage per path: accuracy is not the whole scoreboard

Unit 6 taught you to read **catalog coverage** (the fraction of the catalog that appears in at least one reader's top 10) next to hit@10. Collect both for every path in this notebook -- popularity, item-item, last-1, last-3, last-5, transition, the shuffled transition control, and SASRec -- into one pandas `DataFrame` with columns `path`, `hit@10`, `coverage`, sorted by `hit@10` (highest first). Every number is already computed; reuse it, do not refit or re-board.

Confirm that every recency path covers more of the catalog than item-item (expect last-3 ~0.71 vs ~0.34), and that coverage *rises* as the window shrinks (last-1 > last-3 > last-5). In two or three sentences, explain why querying with **fewer** books surfaces a **wider** range of recommendations across readers, and name the path whose high coverage comes with poor accuracy (so coverage alone is not the goal either).

In [ ]:
coverage_table = None   # TODO: pd.DataFrame([
                        #     {"path": "popularity", "hit@10": popularity_hit, "coverage": popularity_cov},
                        #     {"path": "item-item", "hit@10": item_item_hit, "coverage": item_item_cov},
                        #     ... last-1 / last-3 / last-5 from sweep, transition, transition-shuffled, sasrec ...
                        # ]).sort_values("hit@10", ascending=False)

recency_widens = None       # TODO: bool - sweep[k][1] > item_item_cov for every k in (1, 3, 5)
coverage_rises = None       # TODO: bool - sweep[1][1] > sweep[3][1] > sweep[5][1]

coverage_explanation = "TODO"  # 2-3 sentences: the bag query sums similarity over a long history, so it
                               # drifts toward the same well-connected books for everyone; a few recent
                               # books point each reader at their own neighbourhood, spreading the top-10s;
                               # last-1 has the widest coverage but poor hit@10 -- coverage is not the goal alone

## Exercise 7 - Challenge: decayed recency instead of a hard last-k cutoff

`LastKItemItemPath` uses a **hard** window: the last 5 distinct books count fully, older ones not at all. A softer idea is **decayed recency**: weight the most recent distinct book by 1, the one before by `rate`, the one before that by `rate ** 2`, and so on over the *whole* distinct history, then score every book by the weighted sum of similarity columns. Is a smooth decay better than a hard cutoff?

The scaffold `ScoredPath` (given) wraps any `score_fn(reader_id) -> np.ndarray` (one score per book in `catalog_ids` order, or `None` for an unknown reader) as a scoreboard-ready path that drops seen books and keeps only positive scores. Write `decayed_scores(reader_id, rate)` using the item-item cosine `S = last_k[5].similarity` (same `catalog_ids` order), and score `rate` = 0.5 and 0.7 with `bookrec.run_validation_scoreboard(path, sessions_path, catalog_ids=catalog_ids, k=K)`. Compare with last-5 (`sweep[5][0]`) and report honestly whether decay wins (expect it does **not** beat the hard last-5 window here; the gentler rate only about ties it). In two or three sentences, explain what the result says about how far back the useful signal reaches. (Optional extension: a **second-order** transition table keyed by the last *two* books.)

In [ ]:
class ScoredPath(BaseRetrievalPath):
    # Given scaffold: wraps score_fn(reader_id) -> one score per book (catalog_ids order), or None.
    def __init__(self, name, score_fn):
        super().__init__(name=name, version="1", _fitted=True)
        self.score_fn = score_fn
        self.catalog_array = np.array(catalog_ids)

    def retrieve(self, query, context, k):
        scores = self.score_fn(int(query))
        if scores is None:
            return []
        # Vectorised: keep positive-score, unseen books; a stable argsort of -score keeps tied
        # books in ascending-id order (catalog_ids is sorted), matching the scoreboard tie-break.
        keep = (scores > 0) & ~np.isin(self.catalog_array, list(context.get("seen", ())))
        top = np.argsort(-np.where(keep, scores, -np.inf), kind="stable")[:k]
        candidates = [Candidate(catalog_ids[col], float(scores[col]), self.name) for col in top if keep[col]]
        return self._finish(candidates, k)


S = None   # TODO: last_k[5].similarity   -- the item-item cosine, rows/columns in catalog_ids order


def decayed_scores(reader_id, rate):
    # Weighted sum of similarity columns: most recent distinct book weight 1, then rate, rate**2, ...
    return None   # TODO: history = sequences.get(reader_id); return None if it is empty;
                  #       walk reversed(history) keeping each book's MOST RECENT occurrence once;
                  #       w = np.zeros(len(catalog_ids)); w[col_of[book]] = rate ** age, collecting
                  #       those columns in a list cols; return S[:, cols] @ w[cols]
                  #       (equals S @ w, since w is zero elsewhere, but reads only the history's columns)


decay_hits = None         # TODO: {rate: bookrec.run_validation_scoreboard(
                          #            ScoredPath(f"decayed-{rate}", lambda r, rate=rate: decayed_scores(r, rate)),
                          #            sessions_path, catalog_ids=catalog_ids, k=K).hit_rate_at_k
                          #        for rate in (0.5, 0.7)}
decay_beats_last5 = None  # TODO: bool - max(decay_hits.values()) > sweep[5][0]   (record honestly; expect False)

decay_explanation = "TODO"  # 2-3 sentences: a smooth decay does not beat the hard last-5 window here; the
                            # useful signal is short-range (the last handful of books), so keeping a long
                            # tail at small weights adds stale taste rather than information

## Exercise 8 - Challenge: the next-in-series diagnostic (series as a lens, never a feature)

The session log's strongest order effect is the obvious one: finish volume 2 of a series, read volume 3 next. `load_series(data_dir / "series.csv.gz")` returns `{item_id: (series_id, volume)}` for the books that belong to a series. Use it **only to choose which events to look at** -- no path ever reads it.

A **next-in-series event** is a scored reader whose last train book is volume $v$ of some series and whose next-item target (`next_item`) is volume $v + 1$ of the **same** series. Count the events (expect roughly 150), then measure, on exactly those readers, whether each path's top 10 contains the target, for item-item, transition and last-5. Expect a wide spread (item-item ~0.30, transition ~0.51, last-5 ~0.74). In three or four sentences, explain why the bag model misses next volumes it "knows about", why the order-aware paths catch them without ever seeing the series table, and why using `series` as a model input would be a different (and here, forbidden) experiment.

In [ ]:
series = None           # TODO: load_series(data_dir / "series.csv.gz")   -- diagnostics only

series_events = None    # TODO: sorted readers r in next_item with a non-empty sequences.get(r) where
                        #       last = sequences[r][-1] and target = next_item[r] are both in series,
                        #       series[target][0] == series[last][0] and series[target][1] == series[last][1] + 1
n_series_events = None  # TODO: len(series_events)   -- roughly 150


def series_hit(path):
    # Fraction of series_events whose next volume is in the path's top K (train history = seen).
    return None   # TODO: same loop as next_item_hit (Exercise 4), restricted to series_events


series_item_item = None    # TODO: series_hit(item_item)    -- ~0.30
series_transition = None   # TODO: series_hit(transition)   -- ~0.51
series_last5 = None        # TODO: series_hit(last_k[5])    -- ~0.74
order_catches_series = None  # TODO: bool - series_transition > series_item_item and series_last5 > series_item_item

series_explanation = "TODO"  # 3-4 sentences: the bag query dilutes the just-finished volume among every
                             # older book; recency and transitions weight it, and "volume v -> v+1" is a
                             # common consecutive pair other readers made; the paths learned it from reading
                             # order alone; feeding `series` to a model would test a different (feature) idea

## Exercise 9 - Challenge: blend recency CF with the transition path

Recency CF (last-5) asks "what is *similar* to what you read lately?"; the transition path asks "what do people read *right after* that?". Unit 6's blend combines calibrated scores from several paths. Blend the two with `run_blended_scoreboard({"last-5": last_k[5], "transition": transition}, sessions_path, catalog_ids=catalog_ids, k=K, pool=50)` (equal weights), and read hit@10 and coverage against both single paths.

Expect a **small** lift over the best single path (about 0.579 vs last-5's ~0.572) and a coverage between the two. Record whether the blend beats the best single path, and by how much. In two or three sentences, say whether the lift is worth the second path, and what kind of signal each path contributes that the other lacks.

In [ ]:
blend_result = None      # TODO: run_blended_scoreboard({"last-5": last_k[5], "transition": transition},
                         #          sessions_path, catalog_ids=catalog_ids, k=K, pool=50)
blend_hit, blend_cov = None, None   # TODO: blend_result.hit_rate_at_k, blend_result.catalog_coverage

best_single_hit = None   # TODO: max(sweep[5][0], transition_hit)
blend_lift = None        # TODO: blend_hit - best_single_hit   -- small (~ +0.01)
blend_beats_best = None  # TODO: bool - blend_hit > best_single_hit

blend_explanation = "TODO"  # 2-3 sentences: the lift is small because the two paths overlap heavily (both
                            # key on the last few books); similarity adds "books like these", transitions add
                            # "what comes next" -- a second path is worth it only if the lift survives noise